# Aplicación RAG con NVIDIA NIM en Google Colab

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos, crea un índice Qdrant temporal y publica una interfaz Gradio.

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL pública temporal de Gradio.


> **Entrega · Daniel Muñeton (SI3003).** Esta es mi copia con las tareas del Bloque A aplicadas:
> - **A1** · ejecucion de punta a punta (celda de pruebas al final, con pregunta fuera de corpus).
> - **A2** · corpus propio en `data/company_documents.json` y ruta corregida en la seccion 3.
> - **A3** · prompt *antes* vs *despues* documentado en la seccion 4.
> - **A4** · `build_pipeline` devuelve el `vectorstore` y las fuentes muestran el score de similitud.

## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [ ]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [ ]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")


## 3. Cargar los documentos

Si quieres usar documentos propios, súbelos a `data/company_documents.json` con los campos `id`, `title`, `content` y, opcionalmente, `metadata`. Si el archivo no existe, la aplicación usa un conjunto de demostración.


> **A2.** Uso mi propio corpus (8 documentos de la empresa ficticia *NebulaCRM*) en `data/company_documents.json` y corrijo la ruta tal como pide la guia:
>
> ```diff
> - company_documents = load_documents(Path("company_documents.json"))
> + company_documents = load_documents(Path("data/company_documents.json"))
> ```
>
> En Colab, clona el repo para traer `data/`:
> `!git clone https://github.com/Danimuneton/si3003-workshop-rag-Danimuneton.git`

In [ ]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


# A2 · ruta corregida al corpus propio, con fallback a la raiz de la sesion.
_ruta = Path("data/company_documents.json")
if not _ruta.exists():
    _ruta = Path("company_documents.json")
company_documents = load_documents(_ruta)
print(f"✅ {len(company_documents)} documentos disponibles")


## 4. Construir el pipeline RAG

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa en Qdrant. El índice vive en memoria mientras la sesión de Colab esté activa.


> **A3 · Ingenieria del prompt.** Defino `PROMPT_ANTES` (vago) y `PROMPT_DESPUES` (estricto: solo contexto, cita el titulo de la fuente, admite cuando no sabe). `build_pipeline` usa `PROMPT_DESPUES`. La comparacion antes/despues esta en la celda A3 del final.
>
> **A4.** `build_pipeline` ahora devuelve tambien el `vectorstore`, para poder calcular el score de similitud de cada fuente en la interfaz.

In [ ]:
# A3 · Prompts para comparar (antes vs despues).
PROMPT_ANTES = (
    "Responde la pregunta del usuario usando el contexto."
)
PROMPT_DESPUES = (
    "Eres un asistente empresarial. Responde UNICAMENTE con informacion "
    "del contexto proporcionado. Si el contexto no contiene la respuesta, "
    "di exactamente: 'No tengo esa informacion en los documentos disponibles.' "
    "No inventes datos. Cita entre parentesis el titulo del documento del "
    "que sacas cada dato. Responde en espanol y de forma concisa."
)
SYSTEM_PROMPT = PROMPT_DESPUES  # cambia a PROMPT_ANTES para comparar (A3)


def build_pipeline(documents: List[Dict]):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=700,
    )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=40,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    # La dimensión se obtiene del modelo y no queda codificada manualmente.
    embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    # A4: devolvemos tambien el vectorstore para calcular scores de similitud.
    return pipeline, chunks, vectorstore


print("Construyendo el índice...")
rag_pipeline, indexed_chunks, vectorstore = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")


## 5. Crear la interfaz Gradio

> **A4.** `chat()` calcula el score coseno de cada fuente con `vectorstore.similarity_search_with_relevance_scores` y lo muestra junto al titulo (`- Titulo - score 0.xxx`).

In [ ]:
def chat(question: str, history: list | None):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."

    try:
        result = rag_pipeline.invoke(question)
        answer = result["answer"].strip()
        documents = result["documents"]

        # A4: score de similitud por fuente (unico por titulo, mayor score).
        scored = vectorstore.similarity_search_with_relevance_scores(question, k=3)
        best = {}
        for doc, score in scored:
            title = doc.metadata.get("title", "Sin título")
            if title not in best or score > best[title]:
                best[title] = score
        ordered = sorted(best.items(), key=lambda kv: kv[1], reverse=True)
        sources_md = "**Fuentes recuperadas (score de similitud):**\n" + "\n".join(
            f"- {title} — score {score:.3f}" for title, score in ordered
        ) if ordered else "No se encontraron documentos relevantes."

    except Exception as error:
        answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG empresarial con NVIDIA NIM") as demo:
    gr.Markdown(
        "# RAG empresarial\n"
        "Consulta documentos internos mediante **LangChain + NVIDIA NIM + Qdrant**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {title}" for title in titles))
            gr.Markdown(f"*{len(indexed_chunks)} chunks en el índice*")
            gr.Markdown("---")
            gr.Markdown(
                f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                f"**LLM:** `{GENERATION_MODEL}`  \n"
                "**Vector store:** Qdrant en memoria  \n"
                "**Retrieval:** k=3, threshold=0.3"
            )

        with gr.Column(scale=2):
            # Gradio 6 usa mensajes {role, content} por defecto;
            # el antiguo argumento type="messages" ya no existe.
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Haz una pregunta sobre los documentos...",
                    label="Pregunta",
                    lines=2,
                    scale=4,
                )
                send_button = gr.Button("Enviar", variant="primary", scale=1)

            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["¿Cuál es el precio del plan Enterprise de CloudSync Pro?"],
            ["¿Cuántos días pueden trabajar remotamente los empleados?"],
            ["¿Qué pasa con los reembolsos mayores a $100?"],
            ["¿Qué ocurre durante la primera semana de onboarding?"],
        ],
        inputs=question_box,
        label="Preguntas de ejemplo",
    )

    send_button.click(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    question_box.submit(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])


## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. La aplicación estará disponible mientras esta celda y el runtime de Colab continúen activos.


In [ ]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.


In [ ]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()


## A1 · Prueba de punta a punta (incluye pregunta fuera de corpus)

Invoca el pipeline directamente, sin la UI. Las primeras cuatro preguntas deben acertar con el corpus propio; la ultima esta **fuera del corpus** y con el prompt A3 el modelo debe admitir que no tiene la informacion.

In [ ]:
preguntas_a1 = [
    "¿Cuanto cuesta el plan Enterprise de NebulaCRM?",
    "¿Cuantos dias por semana se puede trabajar remotamente?",
    "¿Que pasa con los reembolsos mayores a $100?",
    "¿Que ocurre durante la primera semana de onboarding?",
    "¿Cual es la politica de mascotas en la oficina?",  # fuera del corpus (A1)
]
for p in preguntas_a1:
    r = rag_pipeline.invoke(p)
    fuentes = sorted({d.metadata.get("title") for d in r["documents"]})
    print("P:", p)
    print("R:", r["answer"].strip())
    print("Fuentes:", fuentes)
    print("-" * 70)

## A3 · Comparacion del prompt (antes vs despues)

Responde la **pregunta trampa** con cada prompt. El *antes* tiende a divagar o inventar; el *despues* reconoce que no tiene la informacion. Pega ambas salidas en `RESPUESTAS.md`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def responder_con(system_text, pregunta):
    docs = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    ).invoke(pregunta)
    contexto = "\n\n---\n\n".join(
        f"[{d.metadata.get('title','Sin título')}]\n{d.page_content}" for d in docs)
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_text),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])
    cadena = prompt | llm | StrOutputParser()
    return cadena.invoke({"context": contexto, "question": pregunta})

# llm se definio dentro de build_pipeline; lo recreamos aqui para la demo.
from langchain_nvidia_ai_endpoints import ChatNVIDIA
llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, top_p=0.95, max_completion_tokens=700)

trampa = "¿Cual es la politica de mascotas en la oficina?"
print("=== PROMPT ANTES ===")
print(responder_con(PROMPT_ANTES, trampa).strip())
print("\n=== PROMPT DESPUES ===")
print(responder_con(PROMPT_DESPUES, trampa).strip())